# 🏅 OBJECTIVE

Identify digits from a dataset of tens of thousands of handwritten images.

# ⤵️ ACQUIRE DATA

In [ ]:
import urllib.request
from scipy.io import loadmat
import pandas as pd
import numpy as np

mnist_alternative_url = "https://github.com/amplab/datascience-sp14/raw/master/lab7/mldata/mnist-original.mat"
mnist_path = '/kaggle/working/mnist-original.mat'

try:
    with open(mnist_path, 'rb') as f:
        pass
except FileNotFoundError:
    urllib.request.urlretrieve(mnist_alternative_url, mnist_path)

minst_mat = loadmat(mnist_path)
train_df = pd.DataFrame(data=minst_mat['data'].T)
train_df['label'] = minst_mat["label"][0].astype(np.int64)

test_df = pd.read_csv("/kaggle/input/digit-recognizer/test.csv")


# ✅ CHECK DATA

In [ ]:
train_df.head()

In [ ]:
train_df.info()

In [ ]:
img_height = 28
img_width = 28
img_channels = 1
num_classes = train_df['label'].nunique()

print(f"Image dimensions: {img_height}x{img_width}x{img_channels}")
print(f"Number of classes: {num_classes}")

In [ ]:
from tensorflow.keras.utils import to_categorical

predictors = train_df.drop(columns=['label']).values.astype('float32') / 255.0
predictors = predictors.reshape(-1, img_height, img_width, img_channels)
target = to_categorical(train_df['label'], num_classes)

Observations:
- The dataset contains 70,000 samples of handwritten digits (0-9)
- Each sample is represented as a 28x28 pixel gray-scale image (one channel), flattened into a 784-dimensional vector
- The 'label' column indicates the digit (0-9) represented by each image

# 🧹 PREPARE DATA

We'll use random rotation and zoom as data augmentation techniques to help the model generalize better by generating more diverse training samples.

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

augmenter = ImageDataGenerator(
    rotation_range=20,
    zoom_range=0.2
)

augmented_iter = augmenter.flow(
    predictors,
    batch_size=predictors.shape[0],
    shuffle=False
)

augmented_images = next(augmented_iter)

predictors_aug = np.concatenate([predictors, augmented_images], axis=0)
target_aug = np.concatenate([target, target], axis=0)

extra_needed = 200000 - predictors_aug.shape[0]

if extra_needed > 0:
    extra_iter = augmenter.flow(
        predictors,
        batch_size=extra_needed,
        shuffle=False
    )
    
    extra_images = next(extra_iter)
    predictors_aug = np.concatenate([predictors_aug, extra_images], axis=0)
    target_aug = np.concatenate([target_aug, target[:extra_needed]], axis=0)

Shuffle the training data to ensure randomness.

In [ ]:
idx = np.random.permutation(predictors_aug.shape[0])
predictors_aug = predictors_aug[idx]
target_aug = target_aug[idx]

In [ ]:
print(f"Augmented predictors shape: {predictors_aug.shape}")
print(f"Augmented target shape: {target_aug.shape}")

# 🛠️ Build Model

We'll use [MobileNetV2](https://keras.io/api/applications/mobilenet/#mobilenetv2-function), a pre-trained convolutional neural network, as the base model for feature extraction. This model is chosen for its size, efficiency and impressive accuracy compared to larger models. MobileNetV2 expects a 3-channel input with images of minimum size 96x96, so we'll need to adapt our single-channel images accordingly with preprocessing layers.

In [ ]:
import tensorflow as tf
from keras.applications import MobileNetV2
from keras.models import Sequential
from keras.layers import Input, Dense, Flatten, Resizing, Lambda, Dropout

base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(96, 96, 3))
base_model.trainable = False

model = Sequential([
    Input(shape=(img_height, img_width, img_channels)),
    Resizing(96, 96),
    Lambda(lambda x: tf.image.grayscale_to_rgb(x)),
    base_model,
    Flatten(),
    Dense(num_classes, activation='softmax')
])

model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1.0), loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

# 🏁 TRAIN MODEL

In [ ]:
from tensorflow.keras.callbacks import ReduceLROnPlateau
from tensorflow.keras.callbacks import EarlyStopping


reduce_lr  = ReduceLROnPlateau(monitor='val_accuracy', factor=0.1, patience=3, min_lr=1e-6)
early_stopping = EarlyStopping(monitor='val_accuracy', patience=10, restore_best_weights=True)

In [ ]:
train_history = model.fit(predictors_aug, target_aug, validation_split=0.2, epochs=100, batch_size=256, callbacks=[reduce_lr, early_stopping], verbose=1)

# 📊 EVALUATE MODEL

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))
plt.plot(train_history.history['accuracy'], label='train accuracy')
plt.plot(train_history.history['val_accuracy'], label='validation accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy (%)')
plt.legend()
plt.show()

# 🎯 PREDICT

In [ ]:
pixel_columns = [col for col in test_df.columns if col.startswith('pixel')]
test_predictors = test_df[pixel_columns].values.astype('float32') / 255.0
test_predictors = test_predictors.reshape(-1, img_height, img_width, img_channels)
test_df['Label'] = model.predict(test_predictors).argmax(axis=1)

In [ ]:
submission = test_df[['Label']]
submission.index = submission.index + 1
submission.index.name = 'ImageId'

sample_submission = pd.read_csv('/kaggle/input/digit-recognizer/sample_submission.csv', index_col='ImageId')

assert submission.shape == sample_submission.shape
assert submission.index.equals(sample_submission.index)

submission.to_csv('/kaggle/working/submission.csv', index=True)